# Trade Discovery Pipeline Setup
This notebook installs dependencies and runs the Global Evolutionary Loop (GEL) pipeline.

In [ ]:
# Install TA-Lib C-Library
!wget http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
%cd ta-lib/
!./configure --prefix=/usr
!make
!make install
%cd /kaggle/working
!rm -rf ta-lib ta-lib-0.4.0-src.tar.gz

# Install Python dependencies
!pip install ta-lib vectorbt gplearn pyarrow

In [29]:
import os
import sys
import shutil

# 1. Force cleanup of nested directories to start fresh
%cd /kaggle/working
if os.path.exists('gplearn-2'):
    print("Cleaning up existing gplearn-2 directory to prevent nesting...")
    shutil.rmtree('gplearn-2')

# 2. Clone the repository specifically on the 'gplearn' branch
print("Cloning repository (branch: gplearn)...")
!git clone -b gplearn https://github.com/ayan1-git/gplearn-2.git

# 3. Automated discovery of the project root
print("Searching for project root...")
src_path = None
for root, dirs, files in os.walk('/kaggle/working/gplearn-2'):
    if 'src' in dirs:
        src_path = os.path.join(root, 'src')
        break

if src_path:
    project_root = os.path.dirname(src_path)
    print(f"Project root found: {project_root}")
    %cd {project_root}
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
else:
    print("❌ 'src' directory not found!")
    !find /kaggle/working/gplearn-2 -maxdepth 3

print(f"Current working directory: {os.getcwd()}")

/kaggle/working
Cleaning up existing gplearn-2 directory to prevent nesting...
Cloning repository (branch: gplearn)...
Cloning into 'gplearn-2'...
remote: Enumerating objects: 680, done.
remote: Counting objects: 100% (680/680), done.
remote: Compressing objects: 100% (308/308), done.
remote: Total 680 (delta 442), reused 603 (delta 365), pack-reused 0 (from 0)
Receiving objects: 100% (680/680), 4.99 MiB | 15.76 MiB/s, done.
Resolving deltas: 100% (442/442), done.
Searching for project root...
Project root found: /kaggle/working/gplearn-2/trade_discovery
/kaggle/working/gplearn-2/trade_discovery
Current working directory: /kaggle/working/gplearn-2/trade_discovery


In [30]:
# Verify data existence and config
import os
import src.config as cfg

print(f"Configured DATAPATH: {cfg.DATAPATH}")
if os.path.exists(cfg.DATAPATH):
    print(f"✅ Data file found: {cfg.DATAPATH}")
    !ls -lh {cfg.DATAPATH}
else:
    print(f"❌ Data file NOT found at {cfg.DATAPATH}")
    print("Full path check:", os.path.abspath(cfg.DATAPATH))
    print("\nDirectory structure (depth 2):")
    !ls -R data/

Configured DATAPATH: data/BANK_NIFTY_30min_4Y .csv
✅ Data file found: data/BANK_NIFTY_30min_4Y .csv
ls: cannot access 'data/BANK_NIFTY_30min_4Y': No such file or directory
ls: cannot access '.csv': No such file or directory


In [31]:
# Run the GEL Pipeline
!python scripts/main_pipeline.py

2026-05-13 05:38:28 | INFO     | __main__ | Loading raw data from data/BANK_NIFTY_30min_4Y .csv
2026-05-13 05:38:28 | INFO     | src.feature_engineering | Building features for 'close' | 12838 rows | ohlc=True.
2026-05-13 05:38:29 | INFO     | src.feature_engineering | Feature matrix built: shape=(12838, 35) | NaN count=13526
2026-05-13 05:38:29 | INFO     | src.talib_features | talib_features built: shape=(12838, 35) | NaN counts (top 5):
talib_price_pos_65    64
talib_tema_dev        57
talib_macd_hist       33
talib_atr_14_28       28
talib_adx_14          27
dtype: int64
2026-05-13 05:38:29 | INFO     | __main__ | NaN counts per column:
ewma_vol_span260          59
ret_norm_1d               59
ret_norm_3d               59
ret_norm_6d               59
ret_norm_13d              59
                          ..
talib_cdl_morningstar      0
talib_cdl_eveningstar      0
talib_cdl_3whitesol        0
talib_cdl_3blackcrows      0
talib_cdl_shootingstar     0
Length: 70, dtype: int64
2026-05